# Q1: SMOTE-Style ROC Utility Benchmark

This notebook starts the Question 1 experiment from `manuscript/experiment_plan.md`:

> Can MIMIC match classical SMOTE-style ROC utility?

The reusable experiment machinery lives in `src/mimic_experiments/q1_smote_roc.py`. This notebook only chooses parameters, calls that module, and displays the resulting tables and plot.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "mimic").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import GenerationPolicy

from mimic_experiments.q1_smote_roc import (
    Q1Config,
    load_q1_result_tables,
    load_q1_dataset,
    manuscript_result_row,
    plot_q1_roc_sweep,
    print_progress_event,
    q1_dataset_registry,
    q1_result_table_manifest,
    q1_result_table_paths,
    run_mimic_roc_sweep,
)


## Experiment Controls

Edit this cell to choose the dataset, runtime profile, and MIMIC generation configuration. Use `RUN_PROFILE = "run_full"` to execute and save the full 10-fold experiment; use `RUN_PROFILE = "view"` to skip fitting and regenerate displays from the saved CSV tables.


In [ ]:
# Dataset and run controls
DATASET = 5  # use a registry row number, or a key such as "adult_mixed"
RUN_PROFILE = "run_full"  # "run_full" or "view"
RANDOM_STATE = 0
N_JOBS = -1  # use all available cores for parallel fold execution
ARTIFACT_DIR = str(PROJECT_ROOT / "manuscript" / "artifacts" / "q1_smote_roc")
CACHE_MODELS = True
WORKER_THREADS = 1  # keep BLAS/torch threads small when N_JOBS > 1

# MIMIC controls
MIMIC_MODE = "identity"
MIMIC_CAPACITY_RUN_FULL = 0.35
MIMIC_FEATURE_N_JOBS = 1  # keep 1 when outer jobs parallelize work; increase when running one job at a time

# MIMIC generation policy controls
POLICY_METHOD = "smote"
POLICY_NEIGHBOUR_MODE = "normal"
POLICY_N_NEIGHBORS = 5
POLICY_LAMBDA_RANGE = (0.0, 1.0)

# ROC sweep controls: fraction of the training-fold minority deficit to generate
DEFICIT_FRACTIONS = (0.0, 0.25, 0.5, 0.75, 1.0)

registry = q1_dataset_registry()
DATASET_KEY = registry.loc[DATASET, "key"] if isinstance(DATASET, int) else DATASET

config = Q1Config(
    dataset_key=DATASET_KEY,
    run_profile=RUN_PROFILE,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    artifact_dir=ARTIFACT_DIR,
    cache_models=CACHE_MODELS,
    worker_threads=WORKER_THREADS,
    mimic_mode=MIMIC_MODE,
    mimic_capacity_run_full=MIMIC_CAPACITY_RUN_FULL,
    mimic_feature_n_jobs=MIMIC_FEATURE_N_JOBS,
    deficit_fractions=DEFICIT_FRACTIONS,
    policy=GenerationPolicy(
        method=POLICY_METHOD,
        neighbour_mode=POLICY_NEIGHBOUR_MODE,
        n_neighbors=POLICY_N_NEIGHBORS,
        lambda_range=POLICY_LAMBDA_RANGE,
    ),
)
config


Q1Config(dataset_key='satimage', run_profile='run_full', random_state=0, mimic_mode='joint', mimic_capacity_run_full=0.35, deficit_fractions=(0.0, 0.25, 0.5, 0.75, 1.0), n_jobs=-1, artifact_dir='/run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/MIMIC/manuscript/artifacts/q1_smote_roc', cache_models=True, worker_threads=1, mimic_feature_n_jobs=1, policy=GenerationPolicy(method='smote', neighbour_mode='normal', n_neighbors=5, lambda_range=(0.0, 1.0), class_conditioned=False, cluster_conditioned=False))

## Dataset Registry

The registry records the Q1 dataset distribution reported by the SMOTE paper and which loaders are available.


In [3]:
display(
    q1_dataset_registry(
        artifact_dir=ARTIFACT_DIR,
        run_profile=RUN_PROFILE,
        mimic_mode=MIMIC_MODE,
        mimic_capacity=config.mimic_capacity,
        policy=config.policy,
        random_state=RANDOM_STATE,
    )
)


,key,dataset,majority,minority,status,experiment
0,pima,Pima,500,268,ready: OpenML data_id=37,complete: current summary csv
1,phoneme,Phoneme,3818,1586,ready: OpenML data_id=1489,complete: current summary csv
2,adult_mixed,Adult,37155,11687,"ready: OpenML adult v2, mixed features",not run
3,adult_numeric,Adult numeric-only,37155,11687,"ready: OpenML adult v2, numeric features only",not run
4,estate,E-state,46869,6351,optional: source needed,not run
5,satimage,Satimage,5809,626,ready: OpenML data_id=182; smallest class vs rest,not run
6,forest_cover,Forest Cover,35754,2747,ready: sklearn covtype classes 3 vs 4,not run
7,oil,Oil,896,41,optional: source needed,not run
8,mammography,Mammography,10923,260,ready: imbalanced-learn if installed; OpenML f...,not run
9,can,Can,435512,8360,optional: source needed,not run


## Load Dataset


In [4]:
if config.should_run_experiment:
    df = load_q1_dataset(config)
    display(df.head())
    display(df["label"].value_counts().rename_axis("label").to_frame("count"))
else:
    df = None
    print("RUN_PROFILE=view: skipping dataset load; saved CSV tables will be loaded below.")


,Aattr,Battr,Cattr,Dattr,Eattr,Fattr,A1attr,B2attr,C3attr,D4attr,...,D22attr,E23attr,F24attr,A25attr,B26attr,C27attr,D28attr,E29attr,F30attr,label
0,0.117596,1.241362,1.184036,0.815302,-0.158561,1.256483,1.193546,0.818486,-0.141965,0.879481,...,0.807707,-0.069968,1.219160,1.250463,0.597678,-0.054291,1.233342,1.262255,0.603258,majority
1,-1.205362,-1.249654,-0.077532,0.444886,-0.895959,-0.447579,-0.786760,-0.554203,-0.364672,0.092157,...,-0.192752,-0.736996,-0.969292,-0.844805,-0.400030,-0.725852,-0.344432,-0.594534,-0.183967,majority
2,0.779075,0.148811,0.042617,-0.243030,0.800057,0.164136,0.053370,-0.448612,0.154978,-0.345245,...,-0.877277,0.671174,-0.006373,-0.425752,-0.662584,0.691889,0.356801,-0.175259,-0.236449,majority
3,1.146564,0.585831,0.342991,0.021553,0.947536,0.601074,0.353416,0.026550,1.788164,1.010702,...,0.281150,1.412317,1.044084,0.532085,0.282612,1.438068,1.058033,0.842981,0.130923,majority
4,-0.764376,-1.162250,-0.137607,0.180303,-0.969698,-1.146681,-0.126658,0.184937,-0.735851,-1.132569,...,-0.192752,-0.885225,-1.231906,-0.784941,-0.347519,-0.875088,-1.220973,-0.774223,-0.551339,majority


,count
label,
majority,5805
minority,625


## Run Or Reuse Q1 Sweep

`run_full` fits MIMIC inside each training fold, saves CSV result tables, and leaves validation folds untouched. `view` skips fitting and uses the CSV filenames reconstructed from `DATASET` and the config.


In [ ]:
if config.should_run_experiment:
    run_mimic_roc_sweep(df, config, progress=print_progress_event)
else:
    print("RUN_PROFILE=view: skipping experiment run.")

print("Q1 result table filenames:")
for table, path in q1_result_table_paths(config).items():
    print(f"{table}: {path}")
display(q1_result_table_manifest(config))


Starting Q1 fold jobs: 0/10 complete (10 worker(s))


## Load Saved Result Tables

Performance summaries and plots below read from the CSV tables saved by the sweep.


In [ ]:
roc_points, q1_summary = load_q1_result_tables(config)

display(q1_summary)
display(roc_points)


## Plot ROC Points


In [ ]:
fig, ax = plot_q1_roc_sweep(roc_points, config)


## Result Template For Manuscript

Fill the published-reference columns from the SMOTE paper once the corresponding dataset/classifier setup is selected.


In [ ]:
display(manuscript_result_row(config, q1_summary))
